In [1]:
import logging
import claytonlib as clayton
from claytonlib.chart import (
    ChartSafariInput,
    chart_safari,
    evaluate_chart,
    evaluate_chart_top_10,
    chart_config,
    STRATEGY_ONLY_BALLS,
    STRATEGY_ONE_MUD,
    STRATEGY_SIX_BAIT,
    CRITERIA_CAPTURE,
    CRITERIA_WONT_FLEE_10_TURNS,
    SlidingWindowSum,
    NormalWindow,
)
from claytonlib.safari import safari_pokemon_by_name
from claytonlib.compass import (
    CompassSafariInput,
    compass_safari,
    compass_config,
)
from claytonlib.machete import (
    machete_one,
    machete_all,
    machete_jane,
    machete_config,
    JaneNode,
)

In [2]:
# --- Logging ---
# INFO shows per-write-cycle timing; DEBUG adds per-turn RNG detail
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)s: %(message)s',
    datefmt='%H:%M:%S',
)
logging.getLogger('claytonlib.machete').setLevel(logging.INFO)

# --- Target ---
POKEMON_NAME = 'metang'
KEY_SEED     = 0xEC1504DC # delay = 1244
setup_delay_seconds = 300 # 5 minutes of setup
max_target_seconds = 3600

pokemon  = safari_pokemon_by_name(POKEMON_NAME)
strategy = STRATEGY_ONLY_BALLS
criteria = CRITERIA_CAPTURE

# --- Chart inputs ---
# setup_delay_seconds: real-world seconds between hitting the key seed and
# the first reachable encounter (seed confirmation, RNG advance, Sweet Scent, etc.)
# max_target_seconds: last second offset to evaluate (defines the chart window)
inputs = ChartSafariInput(
    key_seed             = KEY_SEED,
    setup_delay_seconds  = setup_delay_seconds,
    max_target_seconds   = max_target_seconds,
    strategy             = strategy,
    criteria             = criteria,
    pokemon              = pokemon,
)

# --- chart_config options ---
chart_config().evaluation_frames_per_write_cycle = 60
chart_config().resume_validation_enabled         = False
chart_config().evaluation_chains_per_write_cycle = 50
# --- Evaluation strategies ---
# sigma_frames=12 ≈ 0.4 s timing window (1 frame = 2 delay units)
# eval_strategy = NormalWindow(sigma_frames=12)
eval_strategy = SlidingWindowSum(window=13)  # uniform 2-second window

# --- Compass inputs ---
# Set COMPASS_TARGET_DELAY to a delay value and COMPASS_INITIAL_TIME to the
# matching "Initial Time" from the evaluate_chart output above.
import datetime as dt
COMPASS_TARGET_DELAY = 48944 # 48948                          # e.g. 19232
COMPASS_INITIAL_TIME = dt.datetime(2000, 5, 27, 21, 57, 44)  # e.g. dt.datetime(2000, 1, 15, 12, 4, 42)
COMPASS_WINDOW       = 60                                     # search ±60 delay units (~1 second)

if COMPASS_TARGET_DELAY is not None and COMPASS_INITIAL_TIME is not None:
    compass_inputs = CompassSafariInput.from_chart(
        inputs,
        window        = COMPASS_WINDOW,
        initial_time  = COMPASS_INITIAL_TIME,
        target_delay  = COMPASS_TARGET_DELAY,
        # target_seed = 0x...  # optional: verified against target_delay + initial_time
        evaluation_strategy = STRATEGY_ONLY_BALLS,  # uncomment to show success column
    )

# --- Machete inputs ---
# MACHETE_SEED: the specific seed identified by compass (e.g. 0xCC16BF30)
# MACHETE_PATH: compass-syntax actions already observed before calling machete
#               (e.g. '010' if you threw ball×3 with 0/1/0 shakes). Leave '' if
#               starting from the beginning of the encounter.
# MACHETE_MAX_TURNS: override the global max_turns limit (None = unlimited)
MACHETE_SEED      = 0xCC16BF30  # e.g. from compass output
MACHETE_PATH      = ''          # e.g. '010' for actions already taken
MACHETE_MAX_TURNS = 20        # None = use machete_config() default (50)

In [2]:
chart_safari(inputs)

NameError: name 'chart_safari' is not defined

In [3]:
evaluate_chart_top_10(inputs, eval_strategy)

13:35:00 INFO claytonlib.chart: top10 across 2858 chain(s): 0.149s
13:35:00 INFO claytonlib.chart: evaluate_chart complete in 1.258s


Top 10 (by score)
 #        Score(p)    Delay      Time        Δ Time  Initial Time
-----------------------------------------------------------------
 1      4.4(33.8%)    48948  22:10:59   13m 15.067s  2000-05-27 21:57:44
 2     4.37(33.6%)   216928  22:57:58   59m 54.733s  2000-06-29 21:58:04
 3     4.37(33.6%)   216930  22:57:58   59m 54.767s  2000-06-29 21:58:04
 4     4.37(33.6%)   216932  22:57:58   59m 54.800s  2000-06-29 21:58:04
 5     4.07(31.3%)   206110  21:57:50   56m 54.433s  2000-06-30 21:00:56
 6     4.07(31.3%)   206112  21:57:50   56m 54.467s  2000-06-30 21:00:56
 7     4.07(31.3%)   206114  21:57:50   56m 54.500s  2000-06-30 21:00:56
 8      4.0(30.8%)   119516  21:58:58   32m 51.200s  2000-07-29 21:26:07
 9      4.0(30.8%)   119518  21:58:58   32m 51.233s  2000-07-29 21:26:07
10      4.0(30.8%)   119520  21:58:58   32m 51.267s  2000-07-29 21:26:07

Best 10 (highest score at each successively lower delay)
 #        Score(p)    Delay      Time        Δ Time  Initial T

In [ ]:
print(input("Input"))

In [11]:
compass_safari(compass_inputs)

=== Compass: Safari Zone Seed Identifier ===
  m      Mud, no crit         Metang is angry!
  M / a  Mud, crit (Anger)    Metang is beside itself with anger!
  b      Bait, no crit        Metang is eating!
  B / e  Bait, crit (Eating)  Metang is busy eating!
  0      Ball, 0 shakes       Oh, no! The Pokémon broke free!
  1      Ball, 1 shake        Aww! It appeared to be caught!
  2      Ball, 2 shakes       Aargh! Almost had it!
  3      Ball, 3 shakes       Shoot! It was so close, too!
  C      Captured (ends)      Gotcha! Metang was caught!
  F      Fled (ends)          Metang fled!
  u      Undo last action     —
  ?x     Uncertain result     —
  Spaces and commas in input are ignored.


Seeds: 119 / 119 remaining
Path:  (none)
Balls: 30
   #        Seed    Delay      Δ
   1. 0xCB16BEF4    48884    -60
   2. 0xCB16BEF6    48886    -58
   3. 0xCC16BEF6    48886    -58
   4. 0xCB16BEF8    48888    -56
   5. 0xCC16BEF8    48888    -56



>>  1m0



Seeds: 11 / 119 remaining
Path:  1m0
Balls: 28
   #        Seed    Delay      Δ
   1. 0xCB16BEFA    48890    -54
   2. 0xCC16BEFC    48892    -52
   3. 0xCC16BF1C    48924    -20
   4. 0xCB16BF20    48928    -16
   5. 0xCC16BF20    48928    -16



>>  b1Bm



Seeds: 1 / 119 remaining
Path:  1m0b1Bm
Balls: 27
   #        Seed    Delay      Δ  Success
   1. 0xCC16BF32    48946     +2  no

╔════════════════════╗
║  Seed identified!  ║
║  seed  = 0xCC16BF32║
║  delay = 48946     ║
║  Δ     = +2        ║
║  path  = 1m0b1Bm   ║
╚════════════════════╝


In [ ]:
# machete_one — find the shortest capture path for the identified seed
result = machete_one(pokemon, seed=0xCC16BF1C, path=MACHETE_PATH, max_turns=20)
if result is None:
    print("No capture path found within the turn limit.")
else:
    print(f"Shortest path: {result}  ({len(result)} actions)")

In [9]:
# machete_all — find every capture path for the identified seed
all_paths, truncated = machete_all(pokemon, seed=0xCC16BF32, path=MACHETE_PATH, max_turns=20)
print(f"{len(all_paths)} capture path(s) found, {truncated} branch(es) truncated by turn limit.")
for p in all_paths[:20]:  # show first 20 to avoid flooding output
    print(f"  {p}")
if len(all_paths) > 20:
    print(f"  ... ({len(all_paths) - 20} more)")

20:55:48 DEBUG claytonlib.machete: machete_all depth=0  queue=0  elapsed=0.000s
20:55:48 DEBUG claytonlib.machete: machete_all depth=1  queue=2  elapsed=0.000s
20:55:48 DEBUG claytonlib.machete: machete_all depth=2  queue=8  elapsed=0.001s
20:55:48 DEBUG claytonlib.machete: machete_all depth=3  queue=8  elapsed=0.001s
20:55:48 DEBUG claytonlib.machete: machete_all depth=4  queue=17  elapsed=0.001s
20:55:48 DEBUG claytonlib.machete: machete_all depth=5  queue=53  elapsed=0.001s
20:55:48 DEBUG claytonlib.machete: machete_all depth=6  queue=134  elapsed=0.002s
20:55:48 DEBUG claytonlib.machete: machete_all depth=7  queue=134  elapsed=0.003s
20:55:48 DEBUG claytonlib.machete: machete_all depth=8  queue=224  elapsed=0.004s
20:55:48 DEBUG claytonlib.machete: machete_all depth=9  queue=674  elapsed=0.006s
20:55:48 DEBUG claytonlib.machete: machete_all depth=10  queue=2024  elapsed=0.014s
20:55:48 DEBUG claytonlib.machete: machete_all depth=11  queue=5570  elapsed=0.034s
20:55:48 DEBUG clayton

1870 capture path(s) found, 19621953 branch(es) truncated by turn limit.
  1m0b1Bm1b0m1m1mmC
  1m0b1Bm1b0m1m10mC
  1m0b1Bm1b0m101mmC
  1m0b1Bm1b0mbm2mmC
  1m0b1Bm1b001m1mmC
  1m0b1Bm1b0bmm2mmC
  1m0b1Bm1bbm1m1mmC
  1m0b1Bm1bb0mm2mmC
  1m0b1BmbM1m1m1mmC
  1m0b1BmbM1m1m1mbC
  1m0b1BmbM1m1m10mC
  1m0b1BmbM1m1m1bmC
  1m0b1BmbM1m101mmC
  1m0b1BmbM1m1010mC
  1m0b1BmbM1m1b1mmC
  1m0b1BmbM1mbm2mmC
  1m0b1BmbM1mbm20mC
  1m0b1BmbM1mb11mmC
  1m0b1BmbM101m1mmC
  1m0b1BmbM101m10mC
  ... (1850 more)


In [3]:
# machete_jane — optimal decision tree across all compass candidates
# Requires compass_inputs to be configured above and compass to have narrowed candidates.
# Use _generate_candidates to pull the full compass window, or pass plain seed ints directly.
from claytonlib.compass import _generate_candidates  # internal, for notebook use

compass_candidates = [
    (ctx, seed)
    for ctx, seed, delay in _generate_candidates(compass_inputs)
]
# Alternatively, pass plain seed ints with pokemon:
# compass_candidates = [0xCC16BF30, 0xCC16BF32, ...]
# tree = machete_jane(compass_candidates, pokemon=pokemon, max_turns=MACHETE_MAX_TURNS)

tree = machete_jane(compass_candidates, max_turns=10, interactive=True)


def print_tree(node, indent=0, outcome=None):
    prefix = "  " * indent
    label = f"[{outcome}] " if outcome else ""
    if node is None:
        print(f"{prefix}{label}(none)")
        return
    prob_pct = float(node.probability) * 100
    cap_str = ""
    if node.direct_capture_prob is not None:
        cap_str = f"  capture={float(node.direct_capture_prob)*100:.1f}%"
    print(f"{prefix}{label}{node.action}  p={prob_pct:.1f}%{cap_str}")
    if node.branches:
        for ch, child in sorted(node.branches.items()):
            print_tree(child, indent + 1, outcome=ch)


print_tree(tree)
# print(compass_candidates)

23:03:55 INFO claytonlib.machete: machete_jane: 119 candidate(s), max_turns=10


Jane is considering 0xCB16BEF4...
Jane thinks 0xCB16BEF4 is a dead-end.
Jane is considering 0xCB16BEF6...
Jane thinks 0xCB16BEF6 is a dead-end.
Jane is considering 0xCC16BEF6...
Jane thinks 0xCC16BEF6 is a dead-end.
Jane is considering 0xCB16BEF8...
Jane thinks 0xCB16BEF8 is a dead-end.
Jane is considering 0xCC16BEF8...
Jane thinks 0xCC16BEF8 is a dead-end.
Jane is considering 0xCB16BEFA...
Jane thinks 0xCB16BEFA is a dead-end.
Jane is considering 0xCC16BEFA...
Jane thinks 0xCC16BEFA is a dead-end.
Jane is considering 0xCB16BEFC...
Jane thinks 0xCB16BEFC is a dead-end.
Jane is considering 0xCC16BEFC...
Jane thinks 0xCC16BEFC is a dead-end.
Jane is considering 0xCB16BEFE...
Jane thinks 0xCB16BEFE is a dead-end.
Jane is considering 0xCC16BEFE...
Jane thinks 0xCC16BEFE is a dead-end.
Jane is considering 0xCB16BF00...
Jane thinks 0xCB16BF00 is a dead-end.
Jane is considering 0xCC16BF00...
Jane thinks 0xCC16BF00 is a dead-end.
Jane is considering 0xCB16BF02...
Jane thinks 0xCB16BF02 is a de

23:03:59 INFO claytonlib.machete: machete_jane: paths collected in 4.029s, building tree across 119 candidate(s)...
23:03:59 INFO claytonlib.machete: machete_jane: done in 4.145s total


Jane thinks 0x9216BF6A is a dead-end.
Jane is considering 0xCC16BF6A...
Jane thinks 0xCC16BF6A is a dead-end.
Jane is considering 0x9216BF6C...
Jane thinks 0x9216BF6C is a dead-end.
Jane suggests "Throw ball, this could be our chance..." (8.4% confidence, 0.8% chance to capture)
  * 0    Ball, 0 shakes    Oh, no! The Pokémon broke free!
  * 1    Ball, 1 shake     Aww! It appeared to be caught!
  * 2    Ball, 2 shakes    Aargh! Almost had it!
    3    Ball, 3 shakes    Shoot! It was so close, too!
  * C    Captured          Gotcha! Metang was caught!
    F    Fled              Metang fled!


What happened?  0


...

Jane suggests "Throw ball" (33.3% confidence)
    0    Ball, 0 shakes    Oh, no! The Pokémon broke free!
    1    Ball, 1 shake     Aww! It appeared to be caught!
  * 2    Ball, 2 shakes    Aargh! Almost had it!
    3    Ball, 3 shakes    Shoot! It was so close, too!
    C    Captured          Gotcha! Metang was caught!
    F    Fled              Metang fled!


What happened?  2


This should be it! Throw the ball!
    0    Ball, 0 shakes    Oh, no! The Pokémon broke free!
    1    Ball, 1 shake     Aww! It appeared to be caught!
    2    Ball, 2 shakes    Aargh! Almost had it!
    3    Ball, 3 shakes    Shoot! It was so close, too!
  * C    Captured          Gotcha! Metang was caught!
    F    Fled              Metang fled!


What happened?  C


Gotcha! Metang was caught!
BALL  p=8.4%  capture=0.8%
  [0] BALL  p=8.4%  capture=2.4%
    [0] BALL  p=5.0%
      [0] BALL  p=4.2%
        [2] BAIT  p=33.3%
          [b] BALL  p=50.0%
            [1] BALL  p=100.0%
              [1] MUD  p=100.0%
                [M] BALL  p=100.0%  capture=100.0%
                  [C] CAPTURED  p=100.0%
      [1] BALL  p=20.0%
        [0] BALL  p=50.0%
          [0] BALL  p=50.0%  capture=50.0%
            [C] CAPTURED  p=100.0%
    [1] BALL  p=15.4%
      [0] BALL  p=16.7%
        [0] BALL  p=33.3%
          [2] BALL  p=100.0%  capture=100.0%
            [C] CAPTURED  p=100.0%
      [1] BALL  p=33.3%
        [0] MUD  p=33.3%
          [m] BALL  p=33.3%
            [0] BALL  p=33.3%
              [2] BALL  p=100.0%  capture=100.0%
                [C] CAPTURED  p=100.0%
    [2] BALL  p=20.0%
      [1] BALL  p=100.0%
        [0] BAIT  p=100.0%
          [b] BALL  p=100.0%
            [0] BALL  p=100.0%
              [0] MUD  p=100.0%
                [m]